# Comparison: category-set versions

This notebook compares versioned result packs by metadata, support and artifact
integrity. Trees with different category universes are not compared directly.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.experiment_common import (
    artifact_paths,
    load_artifact_manifest,
    load_category_map,
)
load_dotenv(PROJECT_ROOT / ".env")
CATEGORY_SET_VERSION = os.getenv("DAMICORE_CATEGORY_SET_VERSION", "v2_30")
PATHS = artifact_paths(CATEGORY_SET_VERSION)
COMMON_WORK_ROOT = PATHS.common
NORMALIZED_WORK_ROOT = PATHS.normalized
CASE_FULL_WORK_ROOT = PATHS.case_full
CASE_BALANCED_WORK_ROOT = PATHS.case_balanced
RESULTS_ROOT = PATHS.results

from hypotheses.violence_against_women.scripts.category_sets import CATEGORY_SETS
from hypotheses.violence_against_women.scripts.experiment_common import (
    ARTIFACT_ROOT,
    read_json,
)

VERSION_NAMES = ["v1_14", "v2_30"]
EXPERIMENT_NAMES = ["case_full", "case_balanced", "normalized_categories"]
version_paths = {version: artifact_paths(version) for version in VERSION_NAMES}
manifests = {
    version: load_artifact_manifest(
        paths.common / "artifact-manifest.json",
        category_set_version=version,
    )
    for version, paths in version_paths.items()
}
comparison_dir = ARTIFACT_ROOT / "results" / "category_set_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)


## Verify manifests, result packs and shared dimensions


In [ ]:
integrity_rows = []
metadata_rows = []
for version, paths in version_paths.items():
    manifest = manifests[version]
    category_order = manifest["category_order"]
    category_map = load_category_map(paths.common / "category-map.csv")
    assert category_map["category"].tolist() == category_order
    assert manifest["dimension_count"] == 20
    assert tuple(category_order) == CATEGORY_SETS[version]

    for experiment in EXPERIMENT_NAMES:
        result_dir = paths.results / experiment
        required = [
            "run-summary.json",
            "support.csv",
            "distance-matrix.csv",
            "clusters.csv",
            "tree.nwk",
        ]
        missing = [name for name in required if not (result_dir / name).exists()]
        if missing:
            raise FileNotFoundError(
                f"Missing result files for {version}/{experiment}: {missing}"
            )
        summary = read_json(result_dir / "run-summary.json")
        support = pd.read_csv(result_dir / "support.csv")
        distances = pd.read_csv(result_dir / "distance-matrix.csv", index_col=0)
        clusters = pd.read_csv(result_dir / "clusters.csv")
        checks = {
            "manifest_version": summary.get("category_set_version") == version,
            "manifest_category_count": summary.get("category_count") == len(category_order),
            "manifest_dimension_count": summary.get("dimension_count") == 20,
            "support_categories": set(support["category"]) == set(category_order),
            "distance_categories": (
                distances.index.tolist() == category_order
                and distances.columns.tolist() == category_order
            ),
            "cluster_categories": set(clusters["category"]) == set(category_order),
            "tree_present": (result_dir / "tree.nwk").exists(),
        }
        if not all(checks.values()):
            raise ValueError(f"Incompatible result pack: {version}/{experiment}")
        integrity_rows.append({
            "category_set_version": version,
            "experiment": experiment,
            **checks,
            "integrity_ok": True,
        })
        metadata_rows.append({
            "category_set_version": version,
            "experiment": experiment,
            "category_count": len(category_order),
            "dimension_count": manifest["dimension_count"],
            "support_min": int(support["support"].min()),
            "support_max": int(support["support"].max()),
            "support_total": int(support["support"].sum()),
        })

integrity = pd.DataFrame(integrity_rows)
metadata = pd.DataFrame(metadata_rows)
integrity.to_csv(comparison_dir / "integrity.csv", index=False)
metadata.to_csv(comparison_dir / "metadata-comparison.csv", index=False)
display(metadata)
display(integrity)


## Exact category overlap


In [ ]:
shared_categories = sorted(
    set(manifests["v1_14"]["category_order"])
    & set(manifests["v2_30"]["category_order"])
)
pd.DataFrame({"category": shared_categories}).to_csv(
    comparison_dir / "shared-categories.csv", index=False
)
write_json(
    comparison_dir / "comparison-manifest.json",
    {
        "versions": VERSION_NAMES,
        "experiments": EXPERIMENT_NAMES,
        "shared_category_count": len(shared_categories),
        "direct_tree_distance_comparison": False,
    },
)
print(f"Shared exact categories: {len(shared_categories)}")
